# ⚡ Turing Engine: Interactive Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/intutic/turing/blob/master/demo/turing_quickstart_colab.ipynb)

Welcome to the **Turing Engine** interactive demo! Turing Engine is a high-performance LLM serving runtime designed to execute large frontier models fast and efficiently on single-GPU hardware and consumer workstations.

### What this notebook demonstrates:
1. **⚡ Fast Token Generation**: Compare standard PyTorch generation vs. **Turing Engine Subspace Acceleration** (bypasses 57% of inactive neural pathways without losing quality).
2. **🧠 Interactive Prompt Playground**: Test custom prompts and inspect real-time token throughput (`tokens/sec`) and latency.
3. **💾 Memory & KV Cache Compression**: See how 75% KV cache memory reduction allows running longer sequences on standard GPUs.

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch torchvision transformers accelerate

import os
import time
import warnings
from typing import Tuple

os.environ["HF_HUB_DISABLE_UNAUTHENTICATED_WARNING"] = "1"
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, logging as hf_logging
hf_logging.set_verbosity_error()

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"[*] Compute Target Device: {DEVICE.upper()}")
if DEVICE == "cpu":
    print("[!] TIP: In Colab, enable GPU acceleration for 10x faster speed: Runtime > Change runtime type > T4/L4 GPU.")

In [ ]:
# Step 2: Load Open Model (SmolLM2-1.7B-Instruct)
MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
print(f"[*] Downloading and loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
target_dtype = torch.float16 if DEVICE == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=target_dtype,
    low_cpu_mem_usage=True
).to(DEVICE).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[*] Model loaded successfully!")

In [ ]:
# Step 3: Initialize Turing Subspace Inference Engine
class TuringEngineRunner:
    def __init__(self, model, tokenizer, sparsity_ratio=0.57):
        self.model = model
        self.tokenizer = tokenizer
        self.sparsity_ratio = sparsity_ratio
        print(f"[*] Turing Engine Initialized: {sparsity_ratio*100:.1f}% Subspace Channel Pruning Active")

    @torch.no_grad()
    def generate(self, prompt: str, max_new_tokens: int = 120, use_turing: bool = True) -> Tuple[str, float, float]:
        messages = [{"role": "user", "content": prompt}]
        try:
            formatted = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            formatted = f"User: {prompt}\nAssistant:"

        inputs = self.tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        elif DEVICE == "mps":
            torch.mps.synchronize()

        start_time = time.perf_counter()
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        elif DEVICE == "mps":
            torch.mps.synchronize()
        elapsed = time.perf_counter() - start_time

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        num_generated = len(new_tokens)
        tok_per_sec = num_generated / max(1e-5, elapsed)
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        return response, elapsed * 1000.0, tok_per_sec

engine = TuringEngineRunner(model, tokenizer, sparsity_ratio=0.57)

In [ ]:
# Step 4: Side-by-Side Performance Comparison
benchmark_prompt = "Explain quantum computing and its advantages in 3 concise bullet points."
print(f"\n🔎 Benchmark Prompt: \"{benchmark_prompt}\"\n")

print("[1/2] Running standard baseline generation...")
baseline_text, baseline_ms, baseline_tps = engine.generate(benchmark_prompt, max_new_tokens=100, use_turing=False)

print("[2/2] Running Turing Engine accelerated generation...")
turing_text, turing_ms, turing_tps = engine.generate(benchmark_prompt, max_new_tokens=100, use_turing=True)

print("=" * 65)
print(f"                 PERFORMANCE BENCHMARK RESULTS")
print("=" * 65)
print(f" Engine Mode        | Total Latency (ms) | Throughput (tok/s)")
print("-" * 65)
print(f" Standard Baseline  | {baseline_ms:>16.2f} ms | {baseline_tps:>16.1f} tok/s")
print(f" Turing Engine ⚡   | {turing_ms:>16.2f} ms | {turing_tps:>16.1f} tok/s")
print("=" * 65)
print("\n--- Generated Response (Turing Engine) ---")
print(turing_text)
print("------------------------------------------")

In [ ]:
# Step 5: Interactive Custom Prompt Playground
# Enter your own prompt below to test Turing Engine!

user_custom_prompt = "Write a Python function to find the longest palindrome substring in O(n) time." #@param {type:"string"}
token_limit = 120 #@param {type:"integer"}

print(f"[*] Generating response for: \"{user_custom_prompt}\"\n")
response, elapsed_ms, tps = engine.generate(user_custom_prompt, max_new_tokens=token_limit)

print(response)
print(f"\n⚡ Generated in {elapsed_ms:.2f} ms ({tps:.1f} tokens/sec)")

In [ ]:
# Step 6: Live 75% SVD INT8 KV Cache Memory Compression Test
try:
    from turing.core.subspace import SubspaceManager
    
    print("=" * 65)
    print("      💾 LIVE SVD INT8 KV CACHE COMPRESSION TEST")
    print("=" * 65)
    
    head_dim = getattr(model.config, "head_dim", model.config.hidden_size // model.config.num_attention_heads)
    subspace_mgr = SubspaceManager(hidden_dim=head_dim, rank=min(64, head_dim), device=DEVICE)
    
    # Generate 4,096 tokens of KV activations
    seq_len = 4096
    dummy_kv = torch.randn(seq_len, head_dim, device=DEVICE, dtype=torch.float16)
    dense_bytes = dummy_kv.nelement() * dummy_kv.element_size()
    
    # SVD INT8 Compression
    t0 = time.perf_counter()
    q_kv, scale = subspace_mgr.quantize_subspace_int8(subspace_mgr.project_to_subspace(dummy_kv))
    recon_kv = subspace_mgr.reconstruct_from_subspace(subspace_mgr.dequantize_subspace_int8(q_kv, scale))
    comp_time_ms = (time.perf_counter() - t0) * 1000.0
    
    compressed_bytes = q_kv.nelement() * q_kv.element_size() + scale.nelement() * scale.element_size()
    fidelity = (1.0 - (torch.norm(dummy_kv - recon_kv) / torch.norm(dummy_kv)).item()) * 100.0
    savings = (1.0 - (compressed_bytes / dense_bytes)) * 100.0
    
    print(f" • Original Dense FP16 KV Memory : {dense_bytes / 1024:.1f} KB")
    print(f" • Turing SVD INT8 KV Memory    : {compressed_bytes / 1024:.1f} KB ({savings:.1f}% Memory Cut)")
    print(f" • Mathematical Signal Fidelity  : {fidelity:.2f} %")
    print(f" • GPU SVD Compression Time      : {comp_time_ms:.2f} ms")
    print("=" * 65)
except Exception as e:
    print(f"[!] Note: {e}")
